## Metrics Explanation  

This section provides a detailed explanation of the metrics calculated in this notebook for analyzing movement patterns based on tracking data grouped by **patients (`ID`)** and sperm cells (`tracker_id`).  

### 1. **VCL (Curvilinear Velocity)**  
- **Definition**: VCL represents the average velocity of a tracker along its actual curvilinear path. It measures the distance traveled over time, calculated for each sperm within a patient's dataset.  
- **Mathematical Formula**:  

$$v_{ci} = \frac{\|p_{i+1} - p_i\| + \|p_i - p_{i-1}\|}{2\Delta t}$$
$$\text{VCL} = \frac{1}{N-2} \sum_{i=1}^{N-1} v_{ci}$$

  Where:  
  - $p_i$: Position at frame $i$ (e.g., $(x_{center}, y_{center})$).  
  - $\|p_{i+1} - p_i\|$: Euclidean distance between positions at frames \(i+1\) and \(i\).  
  - $\|p_i - p_{i-1}\|$: Euclidean distance between positions at frames \(i\) and \(i-1\).  
  - $\Delta t$: Time interval between frames.  
  - $N$: Total number of frames for the tracker.

### 2. **VSL (Straight-Line Velocity)**  
- **Definition**: VSL measures the average velocity along a straight line connecting the initial and final positions of a sperm cell. It quantifies the efficiency of movement in terms of directness.  
- **Mathematical Formula**:  
  $$\text{VSL} = \frac{\|p_{\text{end}} - p_{\text{start}}\|}{T}$$  
  Where:  
  - $p_{\text{start}}$: Starting position.  
  - $p_{\text{end}}$: Ending position.  
  - $T$: Total time ($N \cdot \Delta t$).  

### 3. **VAP (Average Path Velocity)**  
- **Definition**: VAP represents the average velocity along the path taken by the sperm cell, providing a measure of its average speed over time.  
- **Mathematical Formula**:  
  $$\text{total\_distance} = \sum_{i=1}^{n-1} \sqrt{(x_i - x_{i-1})^2 + (y_i - y_{i-1})^2}$$  
  $$\text{total\_time} = (n - 1) \times \Delta t$$  
  $$\text{VAP} = \frac{\text{total\_distance}}{\text{total\_time}}$$  
  Where:  
  - $x_i$, $y_i$: Coordinates of the position at frame $i$.  
  - $n$: Total number of frames for the tracker.  
  - $\Delta t$: Time interval between frames.  

### 4. **ALH (Amplitude of Lateral Head Displacement)**  
- **Definition**: ALH quantifies the average deviation of the tracker’s position from its average path. It reflects the lateral movement relative to the trajectory.  
- **Mathematical Formula**:  
  $$\text{ALH} = \frac{1}{N} \sum_{i=1}^{N} \|\bar{p} - p_i\|$$  
  Where:  
  - $\bar{p}$: Mean position of the tracker over all frames (average path center).  
  - $p_i$: Position at frame $i$.  

### 5. **MAD (Mean Angular Displacement)**  
- **Definition**: MAD measures the average angular displacement between three consecutive positions. It is useful for analyzing the directional changes in movement.  
- **Mathematical Formula**:  
  $$\theta_i = \cos^{-1}\left(\frac{(p_i - p_{i-1}) \cdot (p_{i+1} - p_i)}{\|p_i - p_{i-1}\| \cdot \|p_{i+1} - p_i\|}\right)$$  
  $$\text{MAD} = \frac{1}{N-2} \sum_{i=2}^{N-1} |\theta_i|$$  
  Where:  
  - $\theta_i$: Angle between vectors formed by three consecutive points.  
  - $\cdot$: Dot product of two vectors.  
  - $\| \cdot \|$: Magnitude of a vector.  

---

### Notes  
- Metrics are computed for each tracker ID grouped by patient ID.  
- $N$: Total number of frames per sperm tracker.  
- All calculations assume the data is ordered temporally (e.g., by `ID`).  
- Metrics such as VCL, VSL, and VAP are expressed in units of distance per time (e.g., $\mu m/s$), while ALH is expressed in units of distance (e.g., $\mu m$), and MAD is in degrees.

# Imports

In [1]:
import pandas as pd
from nb_utils import set_root
import numpy as np
import sys
import json
import os
from pathlib import Path
from typing import List, Union


PROJECT_DIR = set_root(2)

c:\Users\thifa\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\thifa\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Parameters

In [2]:
path_data = PROJECT_DIR / "data"
path_intermediate = path_data / "02_intermediate"
path_primary = path_data / "03_primary"

file_path_data = path_primary / "tracker.parquet"
file_path_horm = path_intermediate / "data_horm_concat.parquet"
tracker_columns = ["tracker_id",	"class_id",	"x_min",	"y_min",	"x_max",	"y_max",	"x_center",	"y_center"]

# Data

In [3]:
data = pd.read_parquet(file_path_data)
#data_horm = pd.read_parquet(file_path_horm)
data.head()

,tracker_id,class_id,x_min,y_min,x_max,y_max,ID,x_center,y_center
0,0,0,4.348797,118.514236,22.273115,138.625229,47,13.310956,128.569733
1,1,0,485.946716,145.540283,506.476501,166.668884,47,496.211609,156.104584
2,2,0,28.142849,251.524780,49.713924,272.921570,47,38.928387,262.223175
3,3,0,103.487030,363.432861,124.152924,384.810669,47,113.819977,374.121765
4,4,0,81.397095,360.035797,98.433258,378.959076,47,89.915176,369.497437


# Functions

In [ ]:
def set_root(level: int = 1) -> Path:
    for i in range(level):
        if i == 0:
            PROJECT_DIR = Path.cwd().parent
        else:
            PROJECT_DIR = PROJECT_DIR.parent
    sys.path.append(str(PROJECT_DIR))
    return PROJECT_DIR

def find_name_with_prefix(names: List[str], prefix: str) -> Union[None|str]:
    for name in names:
        if name.startswith(prefix):
            return name
    return None

def generate_path_url(content: Union[pd.DataFrame|np.ndarray], path_video: Union[Path|str], target_col: Union[str|int]= "ID"):
    path_url = {}
    for idx in content[target_col].unique():
        files_path = os.listdir(path_video)
        file_name = find_name_with_prefix(files_path, str(idx) + "_")
        if file_name:
            path_url[idx] = str(path_video / file_name)
    return path_url


#Funções Novas
def calculate_vcl(temp, delta_t=0.02):
    if len(temp) < 3:
        return 0
    a = []
    for idx in range(1, len(temp) - 1):
        x_center, y_center = temp[idx]
        x_center_minus, y_center_minus = temp[idx - 1]
        x_center_plus, y_center_plus = temp[idx + 1]
        norm_minus = np.linalg.norm(np.array([x_center, y_center]) - np.array([x_center_minus, y_center_minus]))
        norm_plus = np.linalg.norm(np.array([x_center_plus, y_center_plus]) - np.array([x_center, y_center]))
        numerador = norm_minus + norm_plus
        denominador = 2 * delta_t
        vci = numerador / denominador
        a.append(vci)
    return sum(a) / (len(temp) - 2)

def calculate_vsl(temp):
    if len(temp) == 1:
        return 0
    init_value = temp[0]
    final_value = temp[-1]
    vsl = np.sqrt(((init_value[0] - final_value[0])**2) + ((init_value[1] - final_value[1]) ** 2))
    return vsl

def calculate_vap(temp, delta_t=0.02):
    if len(temp) < 3:
        return 0
    a = []
    for idx in range(1, len(temp) - 1):
        x_center, y_center = temp[idx]
        x_center_minus, y_center_minus = temp[idx-1]
        a.append(np.sqrt(((x_center - x_center_minus)**(2)) + ((y_center - y_center_minus)**(2))))
    total_time = (len(temp) - 1) * delta_t
    return sum(a) / total_time
def calculate_alh(temp):
    mean_position = np.mean(temp, axis=0)
    a = []
    for idx in range(len(temp)):
        a.append(np.linalg.norm(mean_position - temp[idx]))
    alh = sum(a) / len(a)
    return alh

def calculate_mad(temp):
    if len(temp) < 3:
        return 0
    angles = []
    for idx in range(1, len(temp) - 1):
        p_i_minus_1 = np.array(temp[idx - 1])
        p_i = np.array(temp[idx])
        p_i_plus_1 = np.array(temp[idx + 1])

        vector_1 = p_i - p_i_minus_1
        vector_2 = p_i_plus_1 - p_i
        
        dot_product = np.dot(vector_1, vector_2)
        magnitude_1 = np.linalg.norm(vector_1)
        magnitude_2 = np.linalg.norm(vector_2)
        
        if magnitude_1 == 0 or magnitude_2 == 0:
            continue
        
        cos_theta = dot_product / (magnitude_1 * magnitude_2)
        theta = np.arccos(np.clip(cos_theta, -1.0, 1.0))
        angles.append(np.abs(theta))
    
    if len(angles) == 0:
        return 0
    
    mad = sum(angles) / len(angles)
    return mad


def position_values(group):
    x = group['x_center'].mean()
    y = group['y_center'].mean()
    
    return pd.Series({'x': x, 'y': y})



def calculate_metrics(group):
    positions = list(zip(group['x_center'], group['y_center']))
    delta_t = 0.02
    
    vcl = calculate_vcl(positions, delta_t)
    vsl = calculate_vsl(positions)
    vap = calculate_vap(positions, delta_t)
    alh = calculate_alh(positions)
    mad = calculate_mad(positions)
    mean_positions = position_values(group)
    return pd.Series({'x':mean_positions['x'], 'y':mean_positions['y'] ,'VCL': vcl, 'VSL': vsl, 'VAP': vap, "ALH": alh, "MAD": mad})

def create_group_id(df, x):
    df['time'] = df.groupby('tracker_id').cumcount() // x
    #df['group_id'] = df['group_id'] + 1
    return df


#Funções que geram os json
def metrics_dataframe_to_json(metrics_df, path_url_trackeado, path_url_nao_trackeado):
    data = {'metrics': []}
    for patient_id, patient_group in metrics_df.groupby('ID'):
        patient_metrics = {'id': int(patient_id), 'trackers': []}
        for _, row in patient_group.iterrows():
            tracker_metrics = {
                'tracker_id': int(row['tracker_id']),
                'VCL': int(row['VCL']),
                'VSL': int(row['VSL']),
                'VAP': int(row['VAP']),
                'ALH': int(row['ALH']),
                'MAD': int(row['MAD']),
                'video_url_trackeado': path_url_trackeado.get(row['tracker_id'], 'N/A'),  #`video_url_trackeado`
                'video_url_nao_trackeado': path_url_nao_trackeado.get(row['tracker_id'], 'N/A')  # `video_url_nao_trackeado`
            }
            patient_metrics['trackers'].append(tracker_metrics)
        data['metrics'].append(patient_metrics)
    return json.dumps(data, indent=4)



def dataframe_to_json(df):
    """
    Converts the dataframe to the specified JSON format.

    Args:
        df (pd.DataFrame): DataFrame with metrics and positions.

    Returns:
        str: JSON formatted string.
    """
    data = {'individuos': []}
    for patient_id, patient_group in df.groupby('ID'):
        individual = {'id': int(patient_id), 'espermatozoides': []}
        for tracker_id, group in patient_group.groupby('tracker_id'):
            espermatozoide = {'id': int(tracker_id), 'route': [], 'frames': []}
            for _, row in group.iterrows():
                espermatozoide['route'].append({'x': int(row['x']), 'y': int(row['y'])})
                espermatozoide['frames'].append({
                    'x': int(row['x']),
                    'y': int(row['y']),
                    'VCL': int(row['VCL']),
                    'VSL': int(row['VSL']),
                    'VAP': int(row['VAP']),
                    'ALH': int(row['ALH']),
                    'MAD': int(row['MAD'])
                })
            individual['espermatozoides'].append(espermatozoide)
        data['individuos'].append(individual)
    return json.dumps(data, indent=4)

def save_json_to_directory(json_data, filename):
    """
    Saves the JSON data to a specific directory 'visualization/outputs' located 
    one level up from the current working directory.

    Args:
        json_data (str): JSON formatted string.
        filename (str): Name of the JSON file.
    """
    parent_directory = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
    target_directory = os.path.join(parent_directory, 'visualization', 'outputs')
    
    if not os.path.exists(target_directory):
        os.makedirs(target_directory)
    
    file_path = os.path.join(target_directory, filename)
    

    with open(file_path, 'w') as f:
        f.write(json_data)
    
    print(f"JSON saved to {file_path}")


TypeError: unsupported operand type(s) for |: 'NoneType' and 'type'

# Calculate metrics

In [ ]:
#Calculo por janelas de tempo
fps = 50 #temos que as imagens foram captadas a uma velocidade de 50 frames por segundo
#grouped = data.sort_values(['ID', 'tracker_id']).drop('class_id', axis=1).reset_index()
#cria grupos de acordo com o fps que a gente definiu
grouped = data.groupby(['ID', 'tracker_id']).apply(create_group_id, x=fps).reset_index(drop=True)
#Calcula métricas de janela
result = grouped.groupby(['ID', 'tracker_id', 'time']).apply(calculate_metrics).reset_index()
result.head()

C:\Users\thifa\AppData\Local\Temp\ipykernel_21864\3977648234.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = data.groupby(['ID', 'tracker_id']).apply(create_group_id, x=fps).reset_index(drop=True)
C:\Users\thifa\AppData\Local\Temp\ipykernel_21864\3977648234.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = grouped.groupby(['ID', 'tracker_id', 'time']).apply(calculate_metrics).reset_

,ID,tracker_id,time,x,y,VCL,VSL,VAP,ALH,MAD
0,11,0,0,19.624744,349.230927,4.444847,0.396244,4.246974,0.170185,1.622648
1,11,0,1,19.170092,350.122894,6.357584,0.786696,6.321046,0.364607,1.640269
2,11,0,2,18.771601,350.685974,5.756544,1.160259,5.645854,0.337307,1.637049
3,11,0,3,18.753860,351.576843,10.681688,1.733109,10.363380,0.692034,1.763086
4,11,0,4,15.920337,351.670776,12.608208,3.968054,12.631149,1.064150,1.319060


In [ ]:
#Calculo de métricas gerais
metrics_df = data.groupby(['ID', 'tracker_id']).apply(calculate_metrics).reset_index()
metrics_df


C:\Users\thifa\AppData\Local\Temp\ipykernel_21864\27621040.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  metrics_df = data.groupby(['ID', 'tracker_id']).apply(calculate_metrics).reset_index()


,ID,tracker_id,x,y,VCL,VSL,VAP,ALH,MAD
0,11,0,10.518502,366.689636,7.925055,46.730666,7.912447,12.385462,1.612388
1,11,1,563.170044,308.250153,118.391782,31.410082,114.487664,11.002529,0.392927
2,11,2,315.825226,190.106949,9.989262,22.329380,9.906783,5.312230,1.647657
3,11,3,377.321106,233.077713,19.741126,2.434888,19.512503,1.156313,1.789694
4,11,4,124.799263,405.383881,6.212186,4.056885,6.192793,0.615399,1.660100
...,...,...,...,...,...,...,...,...,...
11001,82,367,522.700684,127.398407,126.152130,79.810471,124.244713,17.935374,1.276317
11002,82,368,371.472076,220.286819,43.923797,5.648314,37.131780,0.848048,1.849477
11003,82,369,371.940277,217.061951,32.755247,1.398786,31.269740,1.407853,1.847628
11004,82,370,405.461609,151.586060,9.586027,0.379230,8.812121,0.214093,0.798095


# Data to JSON

In [ ]:
# URLs dos vídeos trackeados
path_video_trackeado = PROJECT_DIR / "data" / "03_primary" / "tracker_video"
path_url_trackeado = generate_path_url(metrics_df, path_video_trackeado)

# URLs dos vídeos não trackeados
path_video_nao_trackeado = PROJECT_DIR / "datasets" / "data" / "visem" / "visem-dataset" / "video_cut"
path_url_nao_trackeado = generate_path_url(metrics_df, path_video_nao_trackeado)

json_dt = metrics_dataframe_to_json(metrics_df, path_url_trackeado, path_url_nao_trackeado)
save_json_to_directory(json_dt, 'metrics_general.json')

#json de acordo com a janela de tempo que foi definida no início
json_data = dataframe_to_json(result)
save_json_to_directory(json_data, 'data_window.json')


JSON saved to c:\Users\thifa\OneDrive\Documentos\Estudos DEV\cin-dataviz\visualization\outputs\metrics_general.json
JSON saved to c:\Users\thifa\OneDrive\Documentos\Estudos DEV\cin-dataviz\visualization\outputs\data_window.json
